### Structured Output

Models can be requested to provide their response in a formate matching a given schema. This is usefull for ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema tupes and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, description and nested structure.

In [4]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023BB0481E10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023BB15C9AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")   
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")


In [16]:
## Using this function we allow model to follow this validation class of pydantic which is Movie in this case

model_with_structure= model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023BB0481E10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023BB15C9AD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of

In [17]:
model.invoke("provide the details about Iception movie?")

AIMessage(content='<think>\nOkay, the user is asking for details about the movie "Iception." Wait, I think they might have misspelled the title. The correct title is "Inception," right? Let me confirm. "Inception" is a 2010 science fiction action film directed by Christopher Nolan. The user might have meant that. Let me check if there\'s another movie called "Iception." I don\'t think so. So, assuming it\'s "Inception," I should proceed with that.\n\nFirst, I need to outline the key points about "Inception." Let\'s start with the basic information: director, release year, genre. Then the plot summary. The movie is about a thief who enters people\'s dreams to steal secrets, and he\'s offered a chance to erase his criminal past by performing the reverse: planting an idea into someone\'s mind. The main character is Dom Cobb, played by Leonardo DiCaprio. He\'s a professional thief who can enter people\'s dreams and steal information. His team is hired by a wealthy businessman to perform an

In [20]:
model_with_structure.invoke("provide the details about Iception movie?")


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message raw output alongside parsed structure

In [23]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year the movie was released")   
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("provide details about the movie Inceptioin")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to fill in those parameters. I know the title is "Inception". The director is Christopher Nolan. It was released in 2010. The rating is probably around 8.8 on IMDb. Let me make sure those are correct. Yep, that\'s right. So I\'ll structure the tool call with those details.\n', 'tool_calls': [{'id': 'h08ss4qav', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 227, 'total_tokens': 385, 'completion_time': 0.269493377, 'completion_tokens_details': {'reasoning_tokens': 110}, 'prompt_time': 0.009011397, 'prompt_tokens_details': None, 'queue_time': 0.053739573, 